# 12 — Alberta director package: surfaces, tiers, clusters, tables (mirror of the Y2Y-wide `19_tiers_and_clusters`; package spec v1.7 on the curated block + the 2026-09-14 team-output rules M4.33)

Zero solves; runs after **11** (and **11b/11c** for the necessity-test columns). Same structure, tables and rulings as
`analyses/y2y/19_director_surfaces`, with the Alberta deviations (AB spec v0.5 §6, M13/M18):

| | Y2Y-wide package | Alberta package |
|---|---|---|
| VERSION | `config.Y2Y_VERSION` (v3.1 = curated block, window targets) | the same switch via `config.ab_paths()`; a package built on another version is archived to `_superseded_<v>_artifact_block/` first |
| votes | the 12 design positions (diagnostics out) | same: F12 |
| band | guarded 5% | **the applied band decided by 11 (rule D-AB13: 5% mirror unless flat, then 2% per D-AB10)** |
| budget | 30% of region | level A only (M9.6): locked + 10,083 km² |
| clusters | ≥ 100 km², complexes within 25 km | **≥ 10 km², complexes within 10 km** (D-AB7) |
| representativeness value / driver | presence of a class rare in the extent+250 km window | same rule on the **Alberta** window (D-AB12; `spec/v3.1/efg_window_footprints.csv`) |
| necessity test (E19) | `pct_adequacy_forced`, adequacy-pin caption ≥ 50% | same, from 11c (`runs_v3.1/ab_l/A/e19_forced*.tif`) |
| alignment overlay | declared IPCA proposals | the Upper Smoky Nature-First zone + SRP planning area (M13.3) |
| T-D4 | ecoregions | Natural Regions / Subregions of Alberta (2005) |

**Acts follow the analysis (parent M4.33 addendum):** Act 0 = where the values are (the prologue), Act 1 = the core, Act 2 = the
value-specific (scenario) tiers, Act 3 = the opportunity landscape; the hinge sits between Acts 2 and 3. The registers' internal
identifiers (`Act 1` = core, `Act 2` = scenario) already use these numbers; `act_v16` = the display label (`dc.ACT_DISPLAY`).
**Record rules mirrored from M4.33:** deck picks = the top-k complexes grouped a second time into REGIONAL clusters (single linkage,
`PICK_LINK_KM` — 30 km here, 3× the 10 km complex link, as the parent's 75 km is 3× its 25 km; D-AB7) and numbered NORTH → SOUTH,
the core first; core components below the 10 km² cluster floor within `SPECK_LINK_KM` = 10 km of a regional cluster JOIN it and are drawn
with it (the parent's rule k, kept at the parent's 10 km: a vicinity-on-the-map distance, not an extent-relative one — D-AB7 disclosure);
"intactness" is "naturalness" in every output; T-D7 consequences = cluster mean value ÷ allocatable-land mean
(`dc.ValueRatios`) with two REFERENCE rows (existing protected areas; the Upper Smoky Nature-First zone's unprotected part — the
IPCA analogue); E17 leave-one-theme-out shifts read from `runs_v3.1/ab_l/A/e17_t3/` when 11b has solved them.


In [1]:
import importlib, json, pathlib, shutil, sys
from types import SimpleNamespace
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio import features as rfeatures
from scipy import ndimage

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
VP = config.ab_paths(); VERSION = VP.version
assert config.EFG_SUBDIR == VP.efg_subdir, f"config.EFG_SUBDIR {config.EFG_SUBDIR!r} does not match VERSION {VERSION}"
SPEC, DATA, AB4, REC = HERE / "spec", HERE / "data", VP.analysis, VP.records
PKG = HERE / "director_package"
# supersede, never delete: a package built on another manifest version is archived before this one is written
if (PKG / "summary.json").exists():
    _old = json.loads((PKG / "summary.json").read_text()).get("version", "v1")     # the 2026-09-08 build carried no version key
    if _old != VERSION:
        _arch = PKG / f"_superseded_{_old}_artifact_block"; _arch.mkdir(exist_ok=True)
        for _item in list(PKG.iterdir()):
            if not _item.name.startswith("_"):
                shutil.move(str(_item), str(_arch / _item.name))
        print(f"archived the {_old} package to {_arch.relative_to(ROOT)}")
for sub in ("geotiffs", "tables", "figures"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
GEO, TAB = PKG / "geotiffs", PKG / "tables"
AB = config.AB_HANDOFF_DIR
RUNS = VP.runs / "A"
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())
SC = json.loads((SPEC / "scenarios_ab_v1.json").read_text()); BLOCKS = SC["_meta"]["blocks"]
MAN_ALL = pd.read_csv(VP.manifest); assert len(MAN_ALL) in (12, 14), len(MAN_ALL)
MAN = dc.package_manifest(MAN_ALL)                     # the 12 design positions (parent v0.15; `role` under v3.1)
FORMS = list(MAN.formulation_id)
S11 = json.loads((REC / "gate_ab4_summary.json").read_text())        # 11's record: the applied band decided by rule D-AB13
assert S11.get("version", "v1") == VERSION, f"11's summary is {S11.get('version', 'v1')}, not {VERSION} -- run 11 first"
print(f"VERSION {VERSION}: {VP.manifest.name}; runs {RUNS.relative_to(ROOT)}; EFG block {config.EFG_SUBDIR} ({len(lc.efg_paths(AB))} features)")

# ---- AB constants (D-AB7, disclosed in M13) --------------------------------------------------------------
BAND = f"g{round(100 * float(S11.get('applied_band_g', 0.02))):02d}"   # the applied band (D-AB13: 5% mirror unless flat, then 2% per D-AB10)
MIN_KM2, COMPLEX_LINK_KM, PICK_LINK_KM = 10, 10, 30   # parent 100 km2 / 25 km / 75 km (deck picks: regional clusters, 3x the complex link)
SPECK_LINK_KM = dc.SPECK_LINK_KM                     # 10 km, as the parent (M4.33 k): sub-floor core components within it join the nearest regional cluster
THR = dc.FREQ_THR
FLOOR_G = dc.FLOOR_G
N_EFG = len(lc.efg_paths(AB))

# ---- the AB grid as a director_core-compatible object -----------------------------------------------------
with rasterio.open(AB / "cost_uniform.tif") as src:
    tr, shp, prof = src.transform, src.shape, src.profile
pu = lc.pu_mask(AB)
with rasterio.open(AB / "mask_protected_areas.tif") as src:
    locked2d = (src.read(1) == 1) & pu
G = SimpleNamespace(pu=pu, locked2d=locked2d, locked=locked2d[pu], disc=~locked2d[pu], n_pu=int(pu.sum()),
                    n_disc=int((~locked2d[pu]).sum()), shape=shp, transform=tr, crs=config.TARGET_CRS, profile=prof,
                    cell_km2=abs(tr.a * tr.e) / 1e6)
G.rows, G.cols = np.where(pu)
assert G.n_pu == EXTENT["n_pu"]
BUDGET_PCT_A = EXTENT["budget_pct_effective"]
SUMMARY = dict(version=VERSION, manifest=VP.manifest.name, efg_block=config.EFG_SUBDIR, n_efg=N_EFG, pick_link_km=PICK_LINK_KM, speck_link_km=SPECK_LINK_KM,
               efg_target_rule=(str(MAN_ALL.efg_target_rule.iloc[0]) if "efg_target_rule" in MAN_ALL.columns else "flat"),
               applied_band_g=float(S11.get("applied_band_g", 0.02)), applied_band_rule=S11.get("applied_band_rule", "D-AB10: g = 2%"),
               n_formulations=len(FORMS), threshold=THR, floor_g=FLOOR_G, min_km2=MIN_KM2, complex_link_km=COMPLEX_LINK_KM,
               band=BAND, level="A", budget_pct=BUDGET_PCT_A, additions_km2=EXTENT["additions_cells"],
               pa_km2=int(G.locked.sum()), pa_pct_of_extent=float(100 * G.locked.sum() / G.n_pu))

# ---- load every package formulation once (guarded + plain at the applied band; anchors; unions; D) ------------
L = SimpleNamespace(f_guard={}, f_plain={}, union_guard={}, union_plain={}, anchors={}, D_guard={}, D_plain={}, cert={}, forms=FORMS, missing=[])
for fid in FORMS:
    cd = RUNS / fid
    A = ec.read_selections(cd / "anchor.tif", G.pu)[0]
    L.anchors[fid] = A
    m_disc = int(A[G.disc].sum())
    for sem, tag in (("guard", f"guard_{BAND}"), ("plain", BAND)):
        cert = pd.read_csv(cd / f"certificates_{tag}.csv"); assert bool(cert.band_ok.all()), f"{fid}/{tag}"
        S = np.vstack([A[None, :], ec.read_selections(cd / f"mga_{tag}.tif", G.pu)])
        getattr(L, f"f_{sem}")[fid] = S.mean(axis=0).astype(np.float32)
        getattr(L, f"union_{sem}")[fid] = S.any(axis=0)
        getattr(L, f"D_{sem}")[fid] = dc._diam(S, G.disc, m_disc)
        if sem == "guard":
            L.cert[fid] = dict(members=len(cert), band_ok=bool(cert.band_ok.all()), dup=int(cert.duplicate.sum()),
                               time_limited=int((cert.status == "TIME_LIMIT").sum()), runtime_min=float(cert.runtime_s.sum() / 60))
cert = pd.DataFrame(L.cert).T; cert["D_plain"] = pd.Series(L.D_plain); cert["D_guard"] = pd.Series(L.D_guard)
print(f"package: {len(FORMS)} elicited positions at g={BAND[1:]}% (guarded = deliverable) | PU {G.n_pu:,} | unprotected {G.n_disc:,} | "
      f"PAs {G.locked.sum():,} km2 = {SUMMARY['pa_pct_of_extent']:.1f}% of the extent")
print(cert.to_string(float_format=lambda v: f"{v:.3f}"))

VERSION v3.1: manifest_v3.1.csv; runs analyses/alberta_prioritization/runs_v3.1/ab_l/A; EFG block iucn_efg_v3 (13 features)
package: 12 elicited positions at g=02% (guarded = deliverable) | PU 85,133 | unprotected 57,161 | PAs 27,972 km2 = 32.9% of the extent
                 members band_ok dup time_limited runtime_min  D_plain  D_guard
s0_ssp585_theta5      50    True   0            0       1.831    1.000    1.000
s1_ssp585_theta5      50    True   0            0       1.712    0.976    0.977
s2_ssp585_theta5      50    True   0            0       1.766    0.997    0.983
s3_ssp585_theta5      50    True   0            0       1.595    1.000    1.000
s4_ssp585_theta2      50    True   0            0       2.057    1.000    1.000
s5_ssp585_theta5      50    True   0            0       1.675    1.000    1.000
s0_ssp245_theta5      50    True   0            0       1.655    1.000    1.000
s1_ssp245_theta5      50    True   0            0       1.522    0.938    0.938
s2_ssp245_theta5    

In [2]:
# ---- surfaces: guarded F (deliverable) + unguarded F + union membership; by refugia future -------------------
Fg, Fp = dc.ensemble(L.f_guard, FORMS), dc.ensemble(L.f_plain, FORMS)
Ug, Up = dc.union_membership(L, FORMS, True), dc.union_membership(L, FORMS, False)
dc.write_tif(G, Fg, GEO / "F_guarded.tif"); dc.write_tif(G, Fp, GEO / "F_unguarded.tif"); dc.write_tif(G, Ug, GEO / "union_membership_guarded.tif")
for fid in FORMS:
    dc.write_tif(G, L.f_guard[fid], GEO / f"f_guarded_{fid}.tif")
LEVELS = {"585": [f for f in FORMS if "ssp585" in f], "245": [f for f in FORMS if "ssp245" in f]}
F_LEV = {lv: dc.ensemble(L.f_guard, fids) for lv, fids in LEVELS.items() if fids}
for lv, F in F_LEV.items():
    dc.write_tif(G, F, GEO / f"F_guarded_ssp{lv}.tif")
core12 = (Fg >= THR) & G.disc
SUMMARY["by_level"] = {}
for lv, F in F_LEV.items():
    c = (F >= THR) & G.disc
    SUMMARY["by_level"][lv] = dict(n=len(LEVELS[lv]), core_km2=int(c.sum()), jaccard_vs_12=dc.jaccard(c, core12),
                                   pct_of_12_inside=float(100 * (c & core12).sum() / max(core12.sum(), 1)))
c5, c2 = [(F_LEV[l] >= THR) & G.disc for l in ("585", "245")]
SUMMARY["by_level"]["jaccard_585_245"] = dc.jaccard(c5, c2); SUMMARY["by_level"]["union_km2"] = int((c5 | c2).sum()); SUMMARY["by_level"]["intersection_km2"] = int((c5 & c2).sum())
print(f"Act 1 by refugia future vs the 12-position core ({int(core12.sum()):,} km2):\n" + json.dumps(SUMMARY["by_level"], indent=1, default=float))
TD2a = dc.band_table(G, {"unguarded": Fp, "guarded": Fg, **{f"SSP{lv} only": F for lv, F in F_LEV.items()}})
TD2a.to_csv(TAB / "T-D2_bands.csv", index=False)
print("\nT-D2 (bands, unprotected land):"); print(TD2a.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
SUMMARY["frequent_km2"] = dict(guarded=int(core12.sum()), unguarded=int(((Fp >= THR) & G.disc).sum()))
SUMMARY["always_km2"] = dict(guarded=int(((Fg >= 0.95) & G.disc).sum()), unguarded=int(((Fp >= 0.95) & G.disc).sum()))
SUMMARY["ever_in_band_pct"] = dict(guarded=float(100 * (Ug[G.disc] > 0).mean()), unguarded=float(100 * (Up[G.disc] > 0).mean()))
p10 = RUNS / "s0_ssp585_theta5" / "mga_g10.tif"
if p10.exists():
    SUMMARY["s0_g10_union_pct"] = float(100 * ec.read_selections(p10, G.pu).any(axis=0)[G.disc].mean())
# 5%-band context for the guardrail sentence (R5/R7: every unprotected cell is in some 5% plan)
u5 = np.mean([ec.read_selections(RUNS / f / "mga_g05.tif", G.pu).any(axis=0) for f in FORMS], axis=0)
SUMMARY["ever_in_5pct_band_pct"] = float(100 * (u5[G.disc] > 0).mean())
D11 = pd.read_csv(AB4 / "tables" / "E11_delta_matrix.csv", index_col=0).loc[FORMS, FORMS]
off = ~np.eye(len(D11), dtype=bool)
SUMMARY["e11_pairs_in_band"] = [int((D11.values[off] <= 0.05 + 1e-9).sum()), int(off.sum())]
print(f"\never in a 2% band: {SUMMARY['ever_in_band_pct']['guarded']:.1f}% of unprotected land | in a 5% band: {SUMMARY['ever_in_5pct_band_pct']:.1f}% | "
      f"E11 {SUMMARY['e11_pairs_in_band'][0]}/{SUMMARY['e11_pairs_in_band'][1]} pairs mutually near-optimal")

Act 1 by refugia future vs the 12-position core (31 km2):
{
 "585": {
  "n": 6,
  "core_km2": 33,
  "jaccard_vs_12": 0.9393939393939394,
  "pct_of_12_inside": 100.0
 },
 "245": {
  "n": 6,
  "core_km2": 30,
  "jaccard_vs_12": 0.90625,
  "pct_of_12_inside": 93.54838709677419
 },
 "jaccard_585_245": 0.8529411764705882,
 "union_km2": 34,
 "intersection_km2": 29
}

T-D2 (bands, unprotected land):
                    band  unguarded km2  unguarded %disc  guarded km2  guarded %disc  SSP585 only km2  SSP585 only %disc  SSP245 only km2  SSP245 only %disc
      never [0.00, 0.05)           2110              3.7         2020            3.5             1864                3.3             2452                4.3
       rare [0.05, 0.30)          51226             89.6        49978           87.4            50455               88.3            48989               85.7
conditional [0.30, 0.70)           3825              6.7         5132            9.0             4809                8.4             

In [3]:
# ---- pooling (force: one map per scenario), act tiers, Act-2 ownership, T-D2 acts ----------------------------------
POOL, rep = dc.pool_scenarios(G, L.f_guard, MAN, force=True)
POOLp, _ = dc.pool_scenarios(G, L.f_plain, MAN, force=True)
rep.to_csv(TAB / "pooling_check.csv", index=False)
print("pooling check (frequent-tier Jaccard between refugia futures; rule pools iff >= 0.80; deck forces one map per scenario):")
print(rep.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
SUMMARY["pooling"] = rep.to_dict(orient="records")

def act_masks(Fx, POOLx, Ux):
    core = (Fx >= THR) & G.disc
    out = {"Act 1 core (F >= 0.70, all formulations)": core}
    claimed = core.copy(); any_sc = np.zeros(G.n_pu, bool)
    for key, f in POOLx.items():
        sid = key.split("@")[0]
        tier = (f >= THR) & G.disc & ~core
        tag = "Act 2" if sid in dc.ACT2_SCENARIOS else "appendix"
        out[f"{tag} {key}: {dc.SCENARIO_LABEL[sid]} (frequent minus core)"] = tier
        if sid in dc.ACT2_SCENARIOS:
            any_sc |= tier
    out["Act 2 any named scenario (union)"] = any_sc; claimed |= any_sc
    out["Act 3 opportunity (in >= 1 band, not above)"] = (Ux > 0) & G.disc & ~claimed
    out["never (no near-optimal plan selects it)"] = (Ux == 0) & G.disc
    return out
AG, AP = act_masks(Fg, POOL, Ug), act_masks(Fp, POOLp, Up)
AG["Act 1 core - BOTH refugia futures (cell-level intersection)"] = c5 & c2
AG["climate-conditional core - SSP585 future only"] = c5 & ~c2
AG["climate-conditional core - SSP245 future only"] = c2 & ~c5
rows = []
for k in AG:
    kp = AP.get(k)
    rows.append({"tier": k, "guarded km2": int(AG[k].sum()), "guarded %disc": 100 * AG[k].sum() / G.n_disc,
                 "unguarded km2": int(kp.sum()) if kp is not None else np.nan, "unguarded %disc": 100 * kp.sum() / G.n_disc if kp is not None else np.nan})
TD2b = pd.DataFrame(rows)
core_m = AG["Act 1 core (F >= 0.70, all formulations)"]
owner = np.zeros(G.n_pu, np.uint8); nq = np.zeros(G.n_pu, np.uint8); best = np.full(G.n_pu, -1.0, np.float32)
for i, sid in enumerate(dc.ACT2_SCENARIOS, 1):
    if sid not in POOL: continue
    q = (POOL[sid] >= THR) & G.disc & ~core_m; nq[q] += 1
    take = q & (POOL[sid] > best); owner[take] = i; best[take] = POOL[sid][take]
owner[nq >= 2] = 5
dc.write_tif(G, owner, GEO / "act2_owner.tif", dtype="uint8", nodata=255)
own_rows = [{"tier": f"Act 2 by scenario - {dc.SCENARIO_LABEL[s]} only", "guarded km2": int((owner == i).sum()), "guarded %disc": 100 * (owner == i).sum() / G.n_disc,
             "unguarded km2": np.nan, "unguarded %disc": np.nan} for i, s in enumerate(dc.ACT2_SCENARIOS, 1)]
own_rows.append({"tier": "Act 2 by scenario - frequent under 2+ named scenarios", "guarded km2": int((owner == 5).sum()), "guarded %disc": 100 * (owner == 5).sum() / G.n_disc, "unguarded km2": np.nan, "unguarded %disc": np.nan})
TD2b = pd.concat([TD2b, pd.DataFrame(own_rows)], ignore_index=True); TD2b.to_csv(TAB / "T-D2_acts.csv", index=False)
SUMMARY["act2_owner_km2"] = {dc.SCENARIO_LABEL[s]: int((owner == i).sum()) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)} | {"2+ scenarios": int((owner == 5).sum())}
print("\nT-D2 (act tiers):"); print(TD2b.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
tiers = np.zeros(G.n_pu, np.uint8)
tiers[AG["Act 3 opportunity (in >= 1 band, not above)"]] = 1; tiers[AG["Act 2 any named scenario (union)"]] = 2; tiers[core_m] = 3
dc.write_tif(G, tiers, GEO / "act_tiers_guarded.tif", dtype="uint8", nodata=255)

pooling check (frequent-tier Jaccard between refugia futures; rule pools iff >= 0.80; deck forces one map per scenario):
scenario  levels  jaccard                                                   decision  freq_km2_585  freq_km2_245
      s0       2    0.818                                                     POOLED            41            39
      s1       2    0.193 POOLED (forced; rule would separate at Jaccard 0.19 < 0.8)           206           701
      s2       2    0.838                                                     POOLED           209           177
      s3       2    0.714 POOLED (forced; rule would separate at Jaccard 0.71 < 0.8)            14            10
      s4       2    0.692 POOLED (forced; rule would separate at Jaccard 0.69 < 0.8)            12            10
      s5       2    0.882                                                     POOLED            34            30

T-D2 (act tiers):
                                                             tier  gu

PosixPath('/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/analyses/alberta_prioritization/director_package/geotiffs/act_tiers_guarded.tif')

In [4]:
# ---- named context: PAs in the extent + the two AOIs (the IPCA analogue); clustering + complexes + picks ----------------
ext = gpd.read_file(DATA / "ab_extent_v1.gpkg").to_crs(G.crs)
pa = gpd.clip(gpd.read_file(config.PA_VECTOR).to_crs(G.crs), ext).dissolve(by="PA_Name").reset_index(); pa["km2"] = pa.area / 1e6
zones = gpd.read_file("/vsizip/" + str(DATA / "aoi" / "US_SRP_Zones.zip") + "/Data/US_SRP_Zones.shp").to_crs(G.crs)
srp = gpd.read_file("/vsizip/" + str(DATA / "aoi" / "US_SRP_PlanningBoundary.zip") + "/Data/US_SRP_PlanningBoundary.shp").to_crs(G.crs)
AOI = gpd.GeoDataFrame({"name": ["Upper Smoky Nature-First zone", "Upper Smoky SRP planning area"],
                        "geometry": [zones[zones.Zone == "Nature First"].union_all(), srp.union_all()]}, crs=G.crs)
AOI.to_file(GEO / "aoi.gpkg", driver="GPKG")
AOI_MASK = {n: rfeatures.rasterize([(g, 1)], out_shape=G.shape, transform=G.transform, fill=0, dtype="uint8").astype(bool) & G.pu
            for n, g in zip(AOI.name, AOI.geometry)}
named = gpd.GeoDataFrame(pd.concat([pa[pa.km2 >= 25][["PA_Name", "geometry"]].rename(columns={"PA_Name": "name"}), AOI], ignore_index=True), geometry="geometry", crs=G.crs)
core2d = dc.to_grid(G, core_m, fill=False, dtype=bool)

lab1, reg1 = dc.clusters(G, Fg, min_km2=MIN_KM2)
reg1.insert(0, "act", "Act 1"); reg1.insert(1, "key", "ensemble")
reg1["name"] = [dc.placeholder_name(G, lab1 == c, named) if k else "" for c, k in zip(reg1.cid, reg1.kept)]
sens = [dc.sensitivity(G, Fg, min_km2=MIN_KM2).assign(act="Act 1", key="ensemble")]
print(f"Act 1: tier {int(core_m.sum()):,} km2 -> {len(reg1)} components, {int(reg1.kept.sum())} >= {MIN_KM2} km2 ({reg1[reg1.kept].km2.sum():,.0f} km2)")
LABELS = {"act1": lab1}
reg1, cx1 = dc.group_complexes(G, lab1, reg1, link_km=COMPLEX_LINK_KM)
REGS, CX, PICKS, number = [reg1], {"ensemble": cx1}, [], 0
def pick_row(number, act, key, c, names):
    return dict(number=number, act=act, key=key, cid=int(c.anchor_cid), cids=";".join(map(str, c.cids)), n_components=int(c.n),
                name=names.get(int(c.anchor_cid), ""), km2=float(c.km2), meanF=float(c.meanF), lat=float(c.lat), lon=float(c.lon))
names1 = dict(zip(reg1.cid.astype(int), reg1["name"]))
for _, c in cx1.head(dc.TOPK_ACT1).iterrows():
    number += 1; PICKS.append(pick_row(number, "Act 1", "ensemble", c, names1))
print(f"Act 1 complexes (single linkage {COMPLEX_LINK_KM} km): {len(cx1)} from {int(reg1.kept.sum())} components; top {dc.TOPK_ACT1}: "
      + ", ".join(f"{int(c.km2):,} km2 ({int(c.n)} comp.)" for _, c in cx1.head(dc.TOPK_ACT1).iterrows()))
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid not in dc.ACT2_SCENARIOS: continue
    lab, reg = dc.clusters(G, f, min_km2=MIN_KM2, subtract2d=core2d)
    reg.insert(0, "act", "Act 2"); reg.insert(1, "key", key)
    reg["name"] = [dc.placeholder_name(G, (lab == c) & ~core2d, named) if k else "" for c, k in zip(reg.cid, reg.kept)]
    reg, cxk = dc.group_complexes(G, lab, reg, link_km=COMPLEX_LINK_KM)
    LABELS[f"act2_{key}"] = lab; REGS.append(reg); CX[key] = cxk
    sens.append(dc.sensitivity(G, f, min_km2=MIN_KM2, subtract2d=core2d).assign(act="Act 2", key=key))
    kept = reg[reg.kept]
    print(f"Act 2 {key:<4} ({dc.SCENARIO_LABEL[sid]}): {len(reg)} components, {len(kept)} kept after core subtraction "
          f"({kept.residual_km2.sum() if len(kept) else 0:,.0f} km2 residual) -> {len(cxk)} complexes")
    namesk = dict(zip(reg.cid.astype(int), reg["name"]))
    for _, c in cxk.head(dc.TOPK_ACT2).iterrows():
        number += 1; PICKS.append(pick_row(number, "Act 2", key, c, namesk))
for lv, F in F_LEV.items():
    labL, regL = dc.clusters(G, F, min_km2=MIN_KM2)
    regL.insert(0, "act", f"Act 1 ({lv})"); regL.insert(1, "key", f"ensemble_{lv}")
    regL["name"] = [dc.placeholder_name(G, labL == c, named) if k else "" for c, k in zip(regL.cid, regL.kept)]
    regL, cxL = dc.group_complexes(G, labL, regL, link_km=COMPLEX_LINK_KM)
    LABELS[f"act1_{lv}"] = labL; REGS.append(regL); CX[f"ensemble_{lv}"] = cxL
    sens.append(dc.sensitivity(G, F, min_km2=MIN_KM2).assign(act=f"Act 1 ({lv})", key=f"ensemble_{lv}"))
    namesL = dict(zip(regL.cid.astype(int), regL["name"]))
    for j, (_, c) in enumerate(cxL.head(dc.TOPK_ACT1).iterrows(), 1):
        PICKS.append(pick_row(f"{lv}-{j}", f"Act 1 ({lv})", f"ensemble_{lv}", c, namesL))
    print(f"Act 1 ({lv}): tier {int(((F >= THR) & G.disc).sum()):,} km2 -> {int(regL.kept.sum())} clusters >= {MIN_KM2} km2")
REG = pd.concat(REGS, ignore_index=True); REG.to_csv(TAB / "cluster_register_all.csv", index=False)
SENS = pd.concat(sens, ignore_index=True); SENS.to_csv(TAB / "cluster_sensitivity.csv", index=False)
# ---- deck picks, second presentational tier (parent M4.33): REGIONAL clusters (single linkage at PICK_LINK_KM) numbered NORTH -> SOUTH;
# the core first (1..n), then each scenario's picks continue the numbering; by-future picks keep their own labels
RAW_PICKS = pd.DataFrame(PICKS); RAW_PICKS.to_csv(TAB / "picks_raw_complexes.csv", index=False)
grouped, number = [], 0
PICK_COLS = ["act", "key", "cid", "cids", "n_components", "name", "km2", "meanF", "lat", "lon", "members"]
core_raw = RAW_PICKS[RAW_PICKS.act == "Act 1"] if len(RAW_PICKS) else pd.DataFrame(columns=PICK_COLS[:-1] + ["number"])
if len(core_raw):
    gp_ = dc.group_picks(G, LABELS["act1"], core_raw, link_km=PICK_LINK_KM)
    gp_ = dc.absorb_complexes(G, LABELS["act1"], gp_, cx1, link_km=PICK_LINK_KM, reg=reg1, speck_km=SPECK_LINK_KM)   # nearby complexes AND sub-floor specks join the nearest regional cluster (M4.33 i, k)
else:
    gp_ = pd.DataFrame(columns=PICK_COLS); print("NO core clusters >= MIN_KM2 at the applied band -- the core is (nearly) empty on this block (H-AB4 outcome); Act 1 has no picks")
for _, r in gp_.iterrows():
    number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
print(f"core picks: {len(core_raw)} complexes -> {len(gp_)} regional clusters (single linkage {PICK_LINK_KM} km), numbered north -> south: "
      + "; ".join(f"{i + 1} = {r.name} ({r.km2:,.0f} km2; from picks {r.members})" for i, r in enumerate(gp_.itertuples())))
for key in POOL:
    sc_raw = RAW_PICKS[(RAW_PICKS.act == "Act 2") & (RAW_PICKS.key == key)] if len(RAW_PICKS) else RAW_PICKS
    if not len(sc_raw):
        continue
    gs = dc.group_picks(G, LABELS[f"act2_{key}"], sc_raw, link_km=PICK_LINK_KM)
    for _, r in gs.iterrows():
        number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
grouped += [dict(members=str(r["number"]), **{k: v for k, v in r.items()}) for r in PICKS if str(r["act"]).startswith("Act 1 (")]
PICKS = pd.DataFrame(grouped, columns=["number"] + PICK_COLS) if not grouped else pd.DataFrame(grouped); PICKS.to_csv(TAB / "picks.csv", index=False)
np.savez_compressed(GEO / "cluster_labels.npz", **LABELS)
print("\ndeck picks (regional clusters, numbered north -> south; the register ships in full):"); print(PICKS.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
gp = GEO / "clusters.gpkg"
if gp.exists(): gp.unlink()
for k, lab in LABELS.items():
    keyname = "ensemble" if k == "act1" else (f"ensemble_{k[5:]}" if k.startswith("act1_") else k.replace("act2_", ""))
    picked = set(int(x) for _, r in PICKS[PICKS.key == keyname].iterrows() for x in str(r.cids).split(";") if str(x).strip()) if len(PICKS) else set()
    regk = REG[(REG.key == keyname) & (REG.kept | REG.cid.isin(picked))]        # kept components + the specks a cluster absorbed
    if len(regk):
        v = dc.vectorize(G, lab, regk.cid.tolist(), simplify_m=500).merge(regk[["cid", "name", "km2", "meanF"]], on="cid")
        v.to_file(gp, layer=k, driver="GPKG")
print(f"wrote {gp.relative_to(ROOT)} ({len(LABELS)} layers; 500 m simplification)")

Act 1: tier 31 km2 -> 8 components, 1 >= 10 km2 (17 km2)
Act 1 complexes (single linkage 10 km): 1 from 1 components; top 6: 17 km2 (1 comp.)
Act 2 s1   (Core-habitat-forward): 29 components, 6 kept after core subtraction (145 km2 residual) -> 5 complexes
Act 2 s2   (Connectivity-forward): 40 components, 4 kept after core subtraction (82 km2 residual) -> 4 complexes
Act 2 s3   (Biodiversity-forward): 3 components, 0 kept after core subtraction (0 km2 residual) -> 0 complexes
Act 2 s4   (Carbon-forward): 3 components, 0 kept after core subtraction (0 km2 residual) -> 0 complexes
Act 1 (585): tier 33 km2 -> 1 clusters >= 10 km2
Act 1 (245): tier 30 km2 -> 1 clusters >= 10 km2
core picks: 1 complexes -> 1 regional clusters (single linkage 30 km), numbered north -> south: 1 = E of White Goat (52.3°N 116.4°W) (23 km2; from picks 1+1specks)

deck picks (regional clusters, numbered north -> south; the register ships in full):
number         act          key  cid  cids  n_components           

In [5]:
# ---- T-D1 register: AB-local block percentiles (naturalness axis; representativeness = EFG-count percentile), drivers, AOIs, tenure, T-D7 ----
VALS = {f: np.nan_to_num(lc._read(AB / f"{f}.tif")[G.pu], nan=0.0) for f in lc.continuous_features()}
pct = {}
for f in sorted({x for d in dc.BLOCK_AXES.values() for x in d}):
    ref = np.sort(VALS[f][G.disc]); pct[f] = (np.searchsorted(ref, VALS[f], side="right") / len(ref)).astype(np.float32)
axes = {ax: sum(w * pct[f] for f, w in members.items()).astype(np.float32) for ax, members in dc.BLOCK_AXES.items()}
efg_paths = lc.efg_paths(AB)
efg = np.stack([np.nan_to_num(lc._read(p)[G.pu], nan=0.0) > 0 for p in efg_paths])
count = efg.sum(axis=0).astype(np.float32); ref = np.sort(count[G.disc])
axes["representativeness"] = (np.searchsorted(ref, count, side="right") / len(ref)).astype(np.float32)
P = SimpleNamespace(axes=axes, pct=pct, efg=efg, efg_names=[p.stem for p in efg_paths], efg_count=count)
theta = config.AUDIT["theta"]
DM = {"m_soc theta-tail": VALS["irrecoverable_carbon_m_soc"] >= theta * VALS["irrecoverable_carbon_m_soc"].mean()}
DM["connectivity spike (top 0.2%)"] = VALS["transboundary_connectivity"] >= np.quantile(VALS["transboundary_connectivity"], 0.998)
k = int(DM["m_soc theta-tail"].sum()); refu = VALS["climate_type_macrorefugia"]
DM["refugia densest (area-matched to the m_soc tail)"] = refu >= np.sort(refu)[-k]
rare = np.zeros(G.n_pu, bool); rarest = np.zeros(G.n_pu, bool); n_rare = n_rarest = 0
WINF = REC / "efg_window_footprints.csv"          # v3.1: rarity judged in the Alberta +250 km window (D-AB12; parent M4.29)
if WINF.exists():
    _w = pd.read_csv(WINF).set_index("feature"); _rare_set = set(_w.index[_w.rare_window])
    rare_key = f"rarest-EFG footprint (rare in the extent+{config.EFG_TARGET_WINDOW_KM} km window: <= {100 * config.EFG_RARE_WINDOW_PCT:g}% of the window)"
else:
    _rare_set = None; rare_key = f"rarest-EFG footprint (<= {100 * dc.RARE_EFG_PCT:g}% of PU each)"      # v1 rule
for i, p in enumerate(efg_paths):
    e = np.nan_to_num(lc._read(p)[G.pu], nan=0.0)
    if lc.leverage_of(e)[1] >= config.AUDIT["rare_cap"]: rare |= e > 0; n_rare += 1
    if ((p.stem in _rare_set) if _rare_set is not None else (efg[i].sum() <= dc.RARE_EFG_PCT * G.n_pu)): rarest |= efg[i]; n_rarest += 1
DM["rare-attainable EFG footprint"] = rare; DM[rare_key] = rarest
# the necessity test (E19, notebook 11c): adequacy-forced cells (capture 1.0 in every member of every design formulation)
E19_TIF = RUNS / "e19_forced.tif"
if E19_TIF.exists():
    with rasterio.open(E19_TIF) as _s: FORCED = _s.read(1)[G.pu]
    with rasterio.open(RUNS / "e19_forced_class.tif") as _s: FORCED_CLS = _s.read(1)[G.pu]
    FORCED_NAMES = {int(k): v for k, v in json.loads((REC / "E19_forced_classes.json").read_text()).items()}
    print(f"E19 forced layer loaded: {int((FORCED == 2).sum()):,} km2 forced in all formulations, {int((FORCED >= 1).sum()):,} in >= 1")
else:
    FORCED = None; print("E19 forced layer absent (run 11b + 11c) -- adequacy columns will be NaN")
print(f"driver masks: m_soc theta-tail {k:,} cells | refugia densest (same area) | spike {int(DM['connectivity spike (top 0.2%)'].sum()):,} | "
      f"rare-attainable EFG footprint {int(rare.sum()):,} ({100 * rare.mean():.0f}% of PU) from {n_rare}/{N_EFG} | rarest {int(rarest.sum()):,} from {n_rarest}")
with rasterio.open(DATA / "derived" / "tenure_class.tif") as s: ten = s.read(1)[G.pu]
near_pa = ndimage.distance_transform_edt(~G.locked2d) <= 5
KEY2LAB = {"ensemble": "act1", **{f"ensemble_{lv}": f"act1_{lv}" for lv in F_LEV}, **{k_: f"act2_{k_}" for k_ in POOL}}
KEY2ACT = {"ensemble": "Act 1", **{f"ensemble_{lv}": f"Act 1 ({lv})" for lv in F_LEV}, **{k_: "Act 2" for k_ in POOL}}
VR = dc.ValueRatios(G, P)      # consequences: mean value in the cluster / mean over Alberta's allocatable land (the parent stack = the AB stack on this grid)
CXROWS = []
picked_cids = set()
for _, r in PICKS[~PICKS.act.str.startswith("Act 1 (")].iterrows():          # the deck picks (regional clusters) first
    cids = [int(c_) for c_ in str(r.cids).split(";")]; picked_cids |= {(r.key, c_) for c_ in cids}
    CXROWS.append(dict(act=r.act, key=r.key, cid=int(r.cid), cids=cids, name=r["name"]))
for key, cx in CX.items():                                                       # then every other kept complex
    regk = REG[REG.key == key].set_index("cid")
    for _, c in cx.iterrows():
        if any((key, int(x)) in picked_cids for x in c.cids):
            continue
        CXROWS.append(dict(act=KEY2ACT[key], key=key, cid=int(c.anchor_cid), cids=list(c.cids), name=regk.loc[int(c.anchor_cid), "name"]))
rows = []
for r in pd.DataFrame(CXROWS).itertuples():
    lab = LABELS[KEY2LAB[r.key]]; m2 = np.isin(lab, r.cids)
    if r.act == "Act 2": m2 = m2 & ~core2d
    m1 = m2[G.pu]; n = int(m1.sum())
    prof = dc.star_profile(P, m1)
    pick = PICKS[(PICKS.act == r.act) & (PICKS.key == r.key) & (PICKS.cid == r.cid)]
    row = dict(number=str(pick.number.iloc[0]) if len(pick) else "", name=r.name, act=r.act, n_components=len(r.cids),
               driving=r.key if r.act == "Act 2" else "all formulations", area_km2=n * G.cell_km2,
               mean_guarded_F=float(Fg[m1].mean()), min_guarded_F=float(Fg[m1].min()))
    row.update({f"pct_{a}": prof[a] for a in dc.STAR_AXES}); row["efg_classes_present"] = dc.efg_classes_present(P, m1)
    row.update({f"ratio_{a}": v for a, v in VR.of(m1).items()})           # consequences: x times the average allocatable cell
    row.update({f"driver_{k_}": 100 * float(v[m1].mean()) for k_, v in DM.items()})
    if FORCED is not None:
        fa = 100 * float((FORCED[m1] == 2).mean()); fy = 100 * float((FORCED[m1] >= 1).mean())
        cls = FORCED_CLS[m1 & (FORCED >= 1)]
        pin = FORCED_NAMES.get(int(np.bincount(cls).argmax()), "") if cls.size else ""
        row.update(pct_adequacy_forced=fa, pct_forced_any_formulation=fy, adequacy_pin=bool(fa >= 50), adequacy_pin_class=pin)
    else:
        row.update(pct_adequacy_forced=np.nan, pct_forced_any_formulation=np.nan, adequacy_pin=False, adequacy_pin_class="")
    row.update(mean_lat=float(dc.latlon(G)[0][m1].mean()), pct_within_5km_of_PA=100 * float(near_pa[m2].mean()),
               pct_in_NFZ=100 * float(AOI_MASK["Upper Smoky Nature-First zone"][m2].mean()),
               pct_in_SRP_area=100 * float(AOI_MASK["Upper Smoky SRP planning area"][m2].mean()),
               pct_private_ranchland=100 * float((ten[m1] == 5).mean()), pct_crown=100 * float(np.isin(ten[m1], [2, 3]).mean()),
               n_formulations_frequent=int(sum(float(L.f_guard[fid][m1].mean() >= THR) >= 0.5 for fid in FORMS)))
    rows.append(row)
TD1 = pd.DataFrame(rows).sort_values(["act", "area_km2"], ascending=[True, False]).reset_index(drop=True)
TD1.insert(3, "act_v16", TD1.act.map(dc.ACT_DISPLAY).fillna(TD1.act))
TD1.insert(5, "driving_label", [dc.SCENARIO_LABEL[d.split("@")[0]] if d != "all formulations" else "all formulations" for d in TD1.driving])
TD1.to_csv(TAB / "T-D1_cluster_register.csv", index=False)
show = ["number", "name", "act_v16", "n_components", "driving_label", "area_km2", "mean_guarded_F", "mean_lat", "pct_in_NFZ", "pct_in_SRP_area",
        "pct_private_ranchland", "n_formulations_frequent", "pct_adequacy_forced", "adequacy_pin_class"] + [c for c in TD1.columns if c.startswith("driver_")]
if FORCED is not None:
    pins = TD1[TD1.adequacy_pin]
    print(f"ADEQUACY PINS (>= 50% of the cluster forced in every formulation): {len(pins)} of {len(TD1)} clusters -- "
          + (", ".join(f"{r['name']} ({r.adequacy_pin_class})" for _, r in pins.iterrows()) if len(pins) else "none"))
print(f"T-D1 (deck picks = regional clusters within {PICK_LINK_KM} km, then the other complexes within {COMPLEX_LINK_KM} km; % columns are shares of cells):")
print(TD1[show].to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
# ---- T-D7 consequences (parent M4.33): the deck clusters' ratios + two REFERENCE rows on the same allocatable-land denominator:
# existing protected areas, and the Upper Smoky Nature-First zone's unprotected part (the IPCA analogue; the SRP planning area beside it)
RAT = TD1[TD1.act.isin(["Act 1", "Act 2"]) & (TD1.number != "")][["number", "name", "act", "driving_label", "area_km2", "mean_guarded_F"] + [f"ratio_{a}" for a in dc.STAR_AXES]].copy()
RAT["number"] = RAT.number.astype(int); RAT = RAT.sort_values("number")
def _ref(name, m1):
    return dict(number=np.nan, name=name, act="reference", driving_label="", area_km2=int(m1.sum()) * G.cell_km2,
                mean_guarded_F=(float(Fg[m1].mean()) if (m1 & G.disc).any() and not (m1 & G.locked).all() else np.nan), **{f"ratio_{a}": v for a, v in VR.of(m1).items()})
NFZ_ADD = AOI_MASK["Upper Smoky Nature-First zone"][G.pu] & ~G.locked
SRP_ADD = AOI_MASK["Upper Smoky SRP planning area"][G.pu] & ~G.locked
REF = pd.DataFrame([_ref("Existing protected areas", G.locked), _ref("Upper Smoky Nature-First zone (unprotected part)", NFZ_ADD),
                    _ref("Upper Smoky SRP planning area (unprotected part)", SRP_ADD)])
RAT = pd.concat([RAT, REF], ignore_index=True); RAT.to_csv(TAB / "T-D7_consequences.csv", index=False)
print("\nT-D7 consequences (mean value in the cluster / mean over allocatable land; reference rows italic in the render):")
print(RAT.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

E19 forced layer loaded: 0 km2 forced in all formulations, 0 in >= 1
driver masks: m_soc theta-tail 4,384 cells | refugia densest (same area) | spike 171 | rare-attainable EFG footprint 39,914 (47% of PU) from 9/13 | rarest 247 from 1
ADEQUACY PINS (>= 50% of the cluster forced in every formulation): 0 of 12 clusters -- none
T-D1 (deck picks = regional clusters within 30 km, then the other complexes within 10 km; % columns are shares of cells):
number                                                    name             act_v16  n_components        driving_label  area_km2  mean_guarded_F  mean_lat  pct_in_NFZ  pct_in_SRP_area  pct_private_ranchland  n_formulations_frequent  pct_adequacy_forced adequacy_pin_class  driver_m_soc theta-tail  driver_connectivity spike (top 0.2%)  driver_refugia densest (area-matched to the m_soc tail)  driver_rare-attainable EFG footprint  driver_rarest-EFG footprint (rare in the extent+250 km window: <= 1% of the window)
     1                        E of Wh

In [6]:
# ---- T-D3 (from 11's T1 record), tier achievement, T-D5 protected baseline, T-D5b enrichment, T-D4 Natural Regions ----
cap = pd.read_csv(AB4 / "tables" / "T1_anchor_captures.csv", index_col=0)
tail = pd.read_csv(AB4 / "tables" / "T1_tail_capture.csv", index_col=0)
lat, _ = dc.latlon(G)
rows = []
for _, r in MAN.iterrows():
    fid = r.formulation_id
    row = dict(formulation=fid, scenario=dc.SCENARIO_LABEL[r.scenario_id], climate=r.climate_level.replace("_2071_2100", ""), value_statement=dc.SCENARIO_STATEMENT[r.scenario_id])
    for b, feats in BLOCKS.items(): row[f"capture_{b}"] = float(cap.loc[fid, feats].mean())
    row["tail_m_soc"] = float(tail.loc[fid, "irrecoverable_carbon_m_soc"]); row["tail_biomass"] = float(tail.loc[fid, "irrecoverable_carbon_biomass"])
    row["anchor_mean_lat"] = float(lat[L.anchors[fid] & G.disc].mean())
    row["frequent_km2_guarded"] = int((L.f_guard[fid][G.disc] >= THR).sum()); row["frequent_km2_unguarded"] = int((L.f_plain[fid][G.disc] >= THR).sum())
    row["D_unguarded"], row["D_guarded"] = L.D_plain[fid], L.D_guard[fid]
    rows.append(row)
TD3 = pd.DataFrame(rows); TD3.to_csv(TAB / "T-D3_scenarios.csv", index=False)
print("T-D3:"); print(TD3.drop(columns="value_statement").to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

core1, sc1, opp1 = core_m, AG["Act 2 any named scenario (union)"], AG["Act 3 opportunity (in >= 1 band, not above)"]
CUM = {"existing PAs": G.locked, "+ Act 1 core": G.locked | core1, "+ Act 2 scenario tiers": G.locked | core1 | sc1, "+ Act 3 opportunity": G.locked | core1 | sc1 | opp1}
rows = []
for t, m in CUM.items():
    for b, feats in BLOCKS.items():
        caps = [float(VALS[f][m].sum() / VALS[f].sum()) for f in feats]
        for f, c_ in zip(feats, caps): rows.append(dict(tier=t, block=b, feature=f, capture=c_))
        rows.append(dict(tier=t, block=b, feature="BLOCK", capture=float(np.mean(caps))))
TA = pd.DataFrame(rows)
ref = TD3[[c for c in TD3.columns if c.startswith("capture_")]].rename(columns=lambda c: c.replace("capture_", "")); ref.index = TD3.formulation
TA_ref = pd.DataFrame({"s0": ref.loc["s0_ssp585_theta5"], "anchor_min": ref.min(), "anchor_max": ref.max()})
TA.to_csv(TAB / "tier_achievement.csv", index=False); TA_ref.to_csv(TAB / "tier_achievement_reference.csv")
print("\ntier achievement (block capture, cumulative tiers incl. locked PAs):")
print(TA[TA.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(list(CUM)).to_string(float_format=lambda v: f"{v:.3f}"))
SUMMARY["tier_area_pct_disc"] = {k_: float(100 * (m & G.disc).sum() / G.n_disc) for k_, m in [("core", core1), ("scenario", sc1), ("opportunity", opp1)]}

# T-D5: what the existing estate already banks (the AB story: 24-71% per value; m_soc target pre-satisfied)
sc0 = SC["S0_balanced"]["targets"]; pa_share = float(G.locked.sum() / G.n_pu); add_share = BUDGET_PCT_A - pa_share
rows = []
for f, v in VALS.items():
    pa_ = float(v[G.locked].sum() / v.sum()); tgt = float(sc0.get(f, 1.0)); s0cap = float(cap.loc["s0_ssp585_theta5", f])
    rows.append(dict(value=f, pct_of_extent_total_in_PAs=100 * pa_, S0_target=tgt, pct_of_target_already_banked=100 * pa_ / tgt,
                     pct_still_needed_from_unprotected_land=100 * max(tgt - pa_, 0), enrichment_existing_PAs=pa_ / pa_share,
                     enrichment_S0_new_half=(s0cap - pa_) / add_share))
TD5 = pd.DataFrame(rows); TD5.to_csv(TAB / "T-D5_protected_baseline.csv", index=False)
n_efg_pa = int(P.efg[:, G.locked].any(axis=1).sum())
SUMMARY["protected_baseline"] = dict(pa_km2=int(G.locked.sum()), pa_pct_of_region=float(100 * pa_share), pa_pct_of_budget=float(100 * pa_share / BUDGET_PCT_A),
                                     efg_present_in_PAs=n_efg_pa, banked_min=float(TD5.pct_of_extent_total_in_PAs.min()), banked_max=float(TD5.pct_of_extent_total_in_PAs.max()))
print("\nT-D5 (existing PAs, locked in; enrichment = capture share / area share; 'new half' = the level-A additions):")
print(TD5.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
banked = {f: float(v[G.locked].sum() / v.sum()) for f, v in VALS.items()}
def tier_enrich(m1):
    a = m1.sum() / G.n_pu
    return {f: float(v[m1].sum() / v.sum()) / a if a > 0 else np.nan for f, v in VALS.items()}
E5 = {}
for _, r in MAN.iterrows():
    fid = r.formulation_id; lab_ = f"{dc.SCENARIO_LABEL[r.scenario_id].split(' (')[0]} {'585' if 'ssp585' in fid else '245'}"
    E5[f"anchor - {lab_}"] = {f: (float(cap.loc[fid, f]) - banked[f]) / add_share for f in VALS}
    E5[f"frequent tier - {lab_}"] = tier_enrich((L.f_guard[fid] >= THR) & G.disc)
E5["frequent tier - ENSEMBLE core"] = tier_enrich(core1); E5["existing PAs"] = {f: banked[f] / pa_share for f in VALS}
TD5b = pd.DataFrame(E5); TD5b.index.name = "value"; TD5b.to_csv(TAB / "T-D5b_enrichment_by_scenario.csv")

# T-D4: Natural Regions / Subregions of Alberta (2005) -- the ecoregion analogue (acquired; REST layer 0)
NSR = gpd.read_file(DATA / "aoi" / "natural_subregions_2005.gpkg").to_crs(G.crs)
zones_id = rfeatures.rasterize(((g, i + 1) for i, g in enumerate(NSR.geometry)), out_shape=G.shape, transform=G.transform, fill=0, dtype="int32")[G.pu]
rows = []
for i, (nr, ns) in enumerate(zip(NSR.NRNAME, NSR.NSRNAME), start=1):
    inz = zones_id == i
    if not inz.any(): continue
    rows.append({"natural_region": nr, "subregion": ns, "PU km2": int(inz.sum()), "protected km2": int((inz & G.locked).sum()),
                 "core km2": int((inz & (tiers == 3)).sum()), "scenario km2": int((inz & (tiers == 2)).sum()),
                 "opportunity km2": int((inz & (tiers == 1)).sum()), "never km2": int((inz & G.disc & (tiers == 0)).sum()), "mean lat": float(lat[inz].mean())})
TD4 = pd.DataFrame(rows).groupby(["natural_region", "subregion"]).agg({"PU km2": "sum", "protected km2": "sum", "core km2": "sum", "scenario km2": "sum",
                                                                          "opportunity km2": "sum", "never km2": "sum", "mean lat": "mean"}).reset_index().sort_values("core km2", ascending=False)
TD4.to_csv(TAB / "T-D4_natural_subregions.csv", index=False)
TD4r = TD4.groupby("natural_region").sum(numeric_only=True).drop(columns="mean lat").sort_values("core km2", ascending=False); TD4r.to_csv(TAB / "T-D4_natural_regions.csv")
SUMMARY["td4"] = "Natural Regions and Subregions of Alberta (2005), NRNAME / NSRNAME"
print("\nT-D4 by Natural Region:"); print(TD4r.to_string())

T-D3:
     formulation                       scenario climate  capture_core_habitat  capture_connectivity  capture_carbon  capture_biodiversity  tail_m_soc  tail_biomass  anchor_mean_lat  frequent_km2_guarded  frequent_km2_unguarded  D_unguarded  D_guarded
s0_ssp585_theta5                       Balanced  ssp585                 0.586                 0.526           0.581                 0.439       0.845         0.931           52.533                    41                       2        1.000      1.000
s1_ssp585_theta5           Core-habitat-forward  ssp585                 0.597                 0.519           0.581                 0.437       0.869         0.931           52.388                   206                     206        0.976      0.977
s2_ssp585_theta5           Connectivity-forward  ssp585                 0.576                 0.535           0.579                 0.437       0.854         0.430           52.608                   209                      39        0.997  

In [7]:
# ---- Act 0 (where the values are), Act 3 (the measured gap), the hinge cross-tab; the E19 partition (parent 19 cell 8) -------------
# Value = the block percentile used for the star axes, thresholded at the top 30% of Alberta's DISCRETIONARY land;
# representativeness votes as presence of a class rare in the Alberta +250 km window (D-AB12; the rare-attainable set is
# reported, not used); naturalness (1 - gHM) is a sixth map, "disclosed, not a driver", outside the 0-5 convergence count.
rare_key_ = [k for k in DM if k.startswith("rarest-EFG")][0]
V = dc.value_layers(G, P, DM[rare_key_])
for t, m in V.masks.items():
    dc.write_tif(G, m.astype(np.uint8), GEO / f"value_top30_{t.replace(' ', '_')}.tif", dtype="uint8", nodata=255)
dc.write_tif(G, V.convergence, GEO / "value_convergence.tif", dtype="uint8", nodata=255)
tier_code = tiers                                     # 0 never / 1 opportunity / 2 scenario / 3 core (cell 3)
high = (V.convergence >= 1) & G.disc
gap = high & (tier_code <= 1)                         # Act 3: high-value land that no tier makes irreplaceable
dc.write_tif(G, np.where(gap, V.convergence, 0).astype(np.uint8), GEO / "value_gap.tif", dtype="uint8", nodata=255)
TD6 = dc.coverage_table(G, V, tier_code)
alt = DM["rare-attainable EFG footprint"] & G.disc
TD6.loc[len(TD6)] = {"theme": f"representativeness — rare-attainable classes (NOT used)", "top-value footprint km2": int(alt.sum()),
                     "% of unprotected land": 100 * alt.sum() / G.n_disc,
                     **{f"% of footprint in {nm}": 100 * float((alt & (tier_code == code)).sum()) / max(alt.sum(), 1) for code, nm in dc.TIER_NAMES.items()}}
TD6.to_csv(TAB / "T-D6_value_coverage.csv", index=False)
print("T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):")
print(TD6.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
def _ta(masks):                                       # share of each block's Alberta value by tier (dc.tier_achievement reads the parent stack)
    rows_ = []
    for tier, m in masks.items():
        for b, feats in BLOCKS.items():
            caps_ = [float(VALS[f][m].sum() / VALS[f].sum()) for f in feats]
            for f, c_ in zip(feats, caps_): rows_.append(dict(tier=tier, block=b, feature=f, capture=c_))
            rows_.append(dict(tier=tier, block=b, feature="BLOCK", capture=float(np.mean(caps_))))
    return pd.DataFrame(rows_)
VS = _ta({"existing PAs": G.locked, "core": core1, "scenario tiers": sc1, "opportunity": opp1, "never": G.disc & (tier_code == 0)})
VSp = VS[VS.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(["existing PAs", "core", "scenario tiers", "opportunity", "never"])
VSp.to_csv(TAB / "T-D6b_value_share_by_tier.csv")
print("\nshare of each block's Alberta value by tier (rows sum to 1 over the extent):"); print(VSp.to_string(float_format=lambda v: f"{v:.3f}"))
XT = dc.crosstab(G, V.convergence, tier_code); XT.to_csv(TAB / "hinge_crosstab.csv")
print("\nhinge cross-tab (km2): value-convergence count x reliability class:"); print(XT.to_string())
conv_km2 = {k: int(((V.convergence == k) & G.disc).sum()) for k in range(6)}
print(f"\nhigh-value land (>= 1 theme): {int(high.sum()):,} km2 = {100 * high.sum() / G.n_disc:.0f}% of unprotected land; "
      f"of it {int(gap.sum()):,} km2 ({100 * gap.sum() / max(high.sum(), 1):.0f}%) sits outside the core and the scenario tiers = Act 3")
# biodiversity: the finding is the product -- capture range over EVERY guarded plan at the applied band
bio = [VALS[f] for f in BLOCKS["biodiversity"]]; bio_tot = [float(v.sum()) for v in bio]
caps = []
for fid in FORMS:
    Sg = np.vstack([L.anchors[fid][None, :], ec.read_selections(RUNS / fid / f"mga_guard_{BAND}.tif", G.pu)])
    caps += [float(np.mean([v[row].sum() / t for v, t in zip(bio, bio_tot)])) for row in Sg]
    del Sg
caps = np.array(caps)
SUMMARY["biodiversity_plan_capture"] = dict(n_plans=int(caps.size), min=float(caps.min()), max=float(caps.max()), median=float(np.median(caps)))
print(f"biodiversity block capture over all {caps.size} guarded plans: {100 * caps.min():.1f}-{100 * caps.max():.1f}% (median {100 * np.median(caps):.1f}%)")
if FORCED is not None:
    conv_cont = sum(V.masks[t].astype(np.uint8) for t in ("core habitat", "connectivity", "biodiversity", "carbon"))
    def partition(m):
        n = max(int(m.sum()), 1); f = m & (FORCED == 2); mc = m & ~f & (conv_cont >= 2)
        return dict(km2=int(m.sum()), forced_pct=100 * f.sum() / n, multi_claim_pct=100 * mc.sum() / n, other_pct=100 * (m & ~f & ~mc).sum() / n)
    PARTS = {"core": core1, **{f"scenario tier: {dc.SCENARIO_LABEL[s]}": (owner == i) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)}, "opportunity": opp1}
    for _, r in PICKS.iterrows():
        lab = LABELS[KEY2LAB[r.key]]; m2 = np.isin(lab, [int(c_) for c_ in str(r.cids).split(";")])
        if r.act == "Act 2": m2 = m2 & ~core2d
        PARTS[f"pick {r.number}: {r['name']}"] = m2[G.pu]
    E19P = pd.DataFrame([{"unit": k_, **partition(m)} for k_, m in PARTS.items()])
    E19P.to_csv(TAB / "E19_partition.csv", index=False)
    print("\nthe necessity test (E19) partition -- forced / multi-claim (>= 2 non-EFG themes top-30%) / other:")
    print(E19P.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    SUMMARY["e19"] = dict(core_forced_pct=float(E19P.iloc[0].forced_pct), n_pins=int(TD1.adequacy_pin.sum()), partition=E19P.to_dict(orient="records"))
SUMMARY["value"] = dict(top=V.top, rare_efg_rule=rare_key_, footprint_km2={t: int(m.sum()) for t, m in V.masks.items()},
                        convergence_km2=conv_km2, high_value_km2=int(high.sum()), gap_km2=int(gap.sum()),
                        gap_pct_of_high_value=float(100 * gap.sum() / max(high.sum(), 1)),
                        value_share_by_tier=VSp.to_dict(), coverage=TD6.to_dict(orient="records"), crosstab=XT.to_dict())


T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):
                                                  theme  top-value footprint km2  % of unprotected land  % of footprint in core  % of footprint in scenario tiers  % of footprint in opportunity  % of footprint in never
                                           core habitat                    17149                   30.0                     0.2                               1.8                           98.0                      0.0
                                           connectivity                    17149                   30.0                     0.2                               1.7                           98.2                      0.0
                                           biodiversity                    17215                   30.1                     0.1                               0.8                           99.1                      0.0
              

In [8]:
# ---- T-D0: the objectives rows for THIS package (13 and 14 render them through director_plot.values_table; the parent's rows
# are the Y2Y-wide ones inside director_plot.values_rows) -- level-A budget, Alberta targets, the 13 curated features, the tenure line
t0 = SC["S0_balanced"]["targets"]; t4 = SC["S4_carbon"]["targets"]; MS = "irrecoverable_carbon_m_soc"
EFG_T = json.loads((REC / "efg_targets.json").read_text())["targets"]; pb = SUMMARY["protected_baseline"]; ADD = SUMMARY["additions_km2"]
VALUES_ROWS_AB = [
 ("PROTECT — wildlife have sufficient core habitat", "Quantity of core habitat", f"Protected land: today's protected areas are locked in ({pb['pa_pct_of_region']:.0f}% of the Alberta extent) and every plan adds {ADD:,} km²",
  "Y2Y protected areas 2025 (IUCN definitions)", f"The budget: the existing estate + {ADD:,} km² of additions (the Y2Y-wide fill rate of unprotected land, D-AB5)", f"{ADD:,} km² of additions"),
 ("", "Quality of core habitat", "Climate refugia: refugial residence time (1 / backward climate velocity), 2071–2100, two emission futures",
  "AdaptWest 2023, CMIP6 backward climate velocity (8-GCM ensemble)", "Core-habitat theme: 25% of the objective in the balanced position; the two futures are separate value positions", "25% of the objective"),
 ("", "Quality of core habitat", "Naturalness: 1 − human modification", "Theobald et al., global human modification (gHM v3)",
  "In every formulation at its baseline weight; it cannot move the answer — disclosed, not a driver", "baseline weight · not a driver"),
 ("", "Biodiversity", "Mammal richness (species per km², area-of-habitat maps, all species)", "Lumbierres et al., AOH species richness (mammals)",
  "Biodiversity theme: 25% (balanced); the two layers weighted equally (mammals cannot move the answer in Alberta — disclosed, D-AB9)", "12.5% of the objective"),
 ("", "Biodiversity", "Bird richness (species per km², area-of-habitat maps, all species)", "Lumbierres et al., AOH species richness (birds)", "", "12.5% of the objective"),
 ("", "Representativeness", f"Presence of {N_EFG} ecosystem functional groups (curated from 27 present under the input pre-screen, rule R0)",
  "IUCN Global Ecosystem Typology, indicative maps (Keith et al. 2022)",
  f"A representation floor, not a weighted theme: rarity-scaled targets of {100*min(EFG_T.values()):.0f}–{100*max(EFG_T.values()):.0f}% per class, rarity judged in the Alberta extent + 250 km",
  f"targets {100*min(EFG_T.values()):.0f}–{100*max(EFG_T.values()):.0f}% per class"),
 ("CONNECT — wildlife corridors connect core habitats", "Quality of connectivity", "Climate corridors: current-flow centrality", "Carroll et al. 2018",
  "Connectivity theme: 25% (balanced); the two layers weighted equally", "12.5% of the objective"),
 ("", "Quality of connectivity", "Habitat connectivity: transboundary omnidirectional current density", "Pither et al. 2023 / O'Brien et al. (transboundary extension)", "", "12.5% of the objective"),
 ("ADDRESS CLIMATE CHANGE — keep carbon out of the air", "Carbon", "Irrecoverable carbon in biomass", "Berman & McDowell, irrecoverable carbon",
  "Carbon theme: 25% (balanced); the two pools split by mass on the Alberta extent", "carbon share · biomass"),
 ("", "Carbon", "Irrecoverable carbon in mineral soil", "Berman & McDowell, irrecoverable carbon",
  f"Security target: {100*t0[MS]:.0f}% of the Alberta total ({100*t4[MS]:.0f}% in the carbon-forward position; the existing estate already holds most of it)",
  f"carbon share · soil · target {100*t0[MS]:.0f}% ({100*t4[MS]:.0f}%)"),
 ("NOT IN THIS ANALYSIS", "Communities · Water · Cost · Tenure", "Bear-smart communities; water; dollars; crown vs private", "—",
  "Tenure is reported after the fact (crown → protection track, private ranchland → OECM track), never optimized; the additions budget stands in for cost", "—"),
]
pd.DataFrame(VALUES_ROWS_AB, columns=["fundamental_objective", "sub_objective", "performance_measure", "source", "in_the_analysis", "metric"]).to_csv(TAB / "T-D0_values_rows.csv", index=False)   # = director_plot.VALUES_COLUMNS + metric
print(f"wrote {(TAB / 'T-D0_values_rows.csv').relative_to(ROOT)} ({len(VALUES_ROWS_AB)} rows; 13/14 render it to the table spec)")


wrote analyses/alberta_prioritization/director_package/tables/T-D0_values_rows.csv (11 rows; 13/14 render it to the table spec)


In [9]:
# ---- AOI / C4 summary for the deck + tenure by tier + summary.json ------------------------------------------------------
rows = []
for n, m2 in AOI_MASK.items():
    m = m2[G.pu] & G.disc
    rows.append(dict(aoi=n, km2_unlocked=int(m.sum()), share_of_unprotected_pct=100 * m.sum() / G.n_disc, core_inside_km2=int((core1 & m).sum()),
                     scenario_inside_km2=int((sc1 & m).sum()), opportunity_inside_km2=int((opp1 & m).sum()), mean_F_inside=float(Fg[m].mean()), mean_F_unprotected=float(Fg[G.disc].mean()),
                     **{f"mean_f_{k_}": float(POOL[k_][m].mean()) for k_ in POOL if k_ in dc.ACT2_SCENARIOS}))
TAOI = pd.DataFrame(rows); TAOI.to_csv(TAB / "T-AOI_alignment.csv", index=False)
print("AOI alignment (alignment, not assignment):"); print(TAOI.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
NAMES = {2: "crown (Green Area)", 3: "crown (White Area, notated)", 4: "private presumed (non-ranch)", 5: "private ranchland (OECM track)", 6: "unclassified"}
TEN = pd.DataFrame([{"tier": lab_, "km2": int(((tiers == code_) & G.disc).sum()), **{nm: int(((tiers == code_) & G.disc & (ten == k_)).sum()) for k_, nm in NAMES.items()}}
                    for code_, lab_ in ((3, "core"), (2, "value-forward"), (1, "opportunity"), (0, "never"))])
TEN.to_csv(TAB / "T-TEN_tenure_by_tier.csv", index=False)
print("\ntenure by tier (km2; private classes over-counted by the crown-lease share, D-AB8):"); print(TEN.to_string(index=False))
SUMMARY["aoi"] = TAOI.to_dict(orient="records"); SUMMARY["tenure_by_tier"] = TEN.to_dict(orient="records")
# E17 leave-one-theme-out shifts on the curated block (11b's E17-T3 cell; parent M4.32) -- a T-D line, no one-pager (M11.4)
E17_DIR = RUNS / "e17_t3"; lat_, _ = dc.latlon(G)
if (E17_DIR / "efg_out" / "run" / "portfolio.tif").exists():
    s0 = L.anchors["s0_ssp585_theta5"] & G.disc; base_lat = float(lat_[s0].mean())
    rows_ = []
    for b in ["core_habitat", "connectivity", "biodiversity", "carbon", "efg"]:
        p_ = E17_DIR / f"{b}_out" / "run" / "portfolio.tif"
        if not p_.exists(): continue
        sel = ec.read_selections(p_, G.pu)[0] & G.disc
        rows_.append(dict(block_out=b, mean_lat=float(lat_[sel].mean()), delta_lat=float(lat_[sel].mean() - base_lat), jaccard_vs_s0=dc.jaccard(sel, s0),
                          basis=f"{VERSION} (curated block; S0 at level A)"))
    E17 = pd.DataFrame(rows_); E17.to_csv(TAB / "E17_shifts.csv", index=False)
    SUMMARY["e17"] = dict(base_lat=base_lat, basis=f"{VERSION}", shifts=E17.to_dict(orient="records"))
    print("E17 leave-one-theme-out (S0, level A):"); print(E17.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
else:
    print("E17-T3 anchors absent (run 11b's E17 cell) -- no shifts recorded")
SUMMARY["forms"] = FORMS; SUMMARY["pool_keys"] = list(POOL)
SUMMARY["ranchland_pool_km2"] = int((ten[G.disc] == 5).sum()); SUMMARY["crown_share_of_unprotected_pct"] = float(100 * np.isin(ten[G.disc], [2, 3]).mean())
(PKG / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nwrote {PKG.relative_to(ROOT)}/summary.json (VERSION {VERSION}, band {BAND}) -- next: 13_figures.ipynb (the record) and 14_director_outputs.ipynb (the presentation set)")

AOI alignment (alignment, not assignment):
                          aoi  km2_unlocked  share_of_unprotected_pct  core_inside_km2  scenario_inside_km2  opportunity_inside_km2  mean_F_inside  mean_F_unprotected  mean_f_s1  mean_f_s2  mean_f_s3  mean_f_s4
Upper Smoky Nature-First zone           436                      0.76                0                   28                     408           0.34                0.18       0.48       0.36       0.25       0.33
Upper Smoky SRP planning area         10028                     17.54                0                   34                    9994           0.19                0.18       0.18       0.19       0.18       0.22

tenure by tier (km2; private classes over-counted by the crown-lease share, D-AB8):
         tier   km2  crown (Green Area)  crown (White Area, notated)  private presumed (non-ranch)  private ranchland (OECM track)  unclassified
         core    31                  29                            0                          